# Phase 1: Data Cleaning & Merging

In this notebook, we load the raw Kaggle datasets, merge them into a unified view, handle missing values, and parse dates. This sets the foundation for our analytical models.

In [1]:
import pandas as pd
import numpy as np
import os

# Set options for pandas
pd.set_option('display.max_columns', None)

## 1. Load the Datasets
We load the core relational tables needed for customer and operational analytics.

In [2]:
raw_dir = '../data/raw/'

orders = pd.read_csv(os.path.join(raw_dir, 'olist_orders_dataset.csv'))
items = pd.read_csv(os.path.join(raw_dir, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(raw_dir, 'olist_customers_dataset.csv'))
sellers = pd.read_csv(os.path.join(raw_dir, 'olist_sellers_dataset.csv'))
reviews = pd.read_csv(os.path.join(raw_dir, 'olist_order_reviews_dataset.csv'))
products = pd.read_csv(os.path.join(raw_dir, 'olist_products_dataset.csv'))
translation = pd.read_csv(os.path.join(raw_dir, 'product_category_name_translation.csv'))

print(f"Orders: {orders.shape}")
print(f"Items: {items.shape}")
print(f"Customers: {customers.shape}")

Orders: (99441, 8)
Items: (112650, 7)
Customers: (99441, 5)


## 2. Translate Product Categories
Convert Portuguese category names to English.

In [3]:
products = products.merge(translation, on='product_category_name', how='left')
# Drop the portuguese column and rename the english one
products.drop('product_category_name', axis=1, inplace=True)
products.rename(columns={'product_category_name_english': 'product_category_name'}, inplace=True)

## 3. Merge the Data
Merge the datasets around the `order_id`.

In [4]:
# Start with orders
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(items, on='order_id', how='left')
df = df.merge(sellers, on='seller_id', how='left')
df = df.merge(products, on='product_id', how='left')

# Reviews can have multiple entries per order. We keep the most recent review per order to avoid duplicates.
reviews_dedup = reviews.sort_values('review_creation_date').drop_duplicates('order_id', keep='last')
df = df.merge(reviews_dedup, on='order_id', how='left')

print(f"Merged Dataset Shape: {df.shape}")

Merged Dataset Shape: (113425, 35)


## 4. Datetime Formatting & Feature Engineering
Convert string dates to `datetime` objects, and calculate operational KPIs like delivery delay.

In [5]:
datetime_cols = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 
    'order_estimated_delivery_date', 'review_creation_date', 
    'review_answer_timestamp'
]

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col])

# Calculate delay: Actual Delivery - Estimated Delivery
# If delay > 0, it was delivered late.
df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.total_seconds() / (24 * 3600)

# Flag delayed orders
df['is_delayed'] = df['delivery_delay_days'] > 0

df[['order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'is_delayed']].head()

,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_delayed
0,2017-10-10 21:25:13,2017-10-18,-7.107488,False
1,2018-08-07 15:27:45,2018-08-13,-5.355729,False
2,2018-08-17 18:06:29,2018-09-04,-17.245498,False
3,2017-12-02 00:28:42,2017-12-15,-12.980069,False
4,2018-02-16 18:17:02,2018-02-26,-9.238171,False


## 5. Handling Missing Values
Identify columns with missing values and handle them appropriately.

In [6]:
missing_percentages = df.isnull().sum() / len(df) * 100
print("Missing Values (>0%):")
print(missing_percentages[missing_percentages > 0].sort_values(ascending=False))

# For items where product_id/seller_id is null (e.g. cancelled orders before fulfillment)
# We will keep them for overall revenue tracking but they won't have product details.
# Review scores that are missing can be filled with median or left as NaN.
# We will leave them as NaN so they are ignored in averages rather than skewed.


Missing Values (>0%):
review_comment_title             88.092572
review_comment_message           57.748292
delivery_delay_days               2.846815
order_delivered_customer_date     2.846815
product_category_name             2.117699
product_name_lenght               2.096540
product_photos_qty                2.096540
product_description_lenght        2.096540
order_delivered_carrier_date      1.735067
review_id                         0.847256
review_answer_timestamp           0.847256
review_creation_date              0.847256
review_score                      0.847256
product_height_cm                 0.699140
product_weight_g                  0.699140
product_length_cm                 0.699140
product_width_cm                  0.699140
order_item_id                     0.683271
product_id                        0.683271
freight_value                     0.683271
seller_id                         0.683271
seller_zip_code_prefix            0.683271
shipping_limit_date             

## 6. Export Processed Data
Save the master cleaned dataset to `data/processed/`.

In [7]:
output_path = '../data/processed/merged_olist_data.csv'
df.to_csv(output_path, index=False)
print(f"Data successfully exported to {output_path}")

Data successfully exported to ../data/processed/merged_olist_data.csv
